In [1]:
import pandas as pd 

df = pd.read_csv("saas_subscriptions.csv")


In [2]:
df.head()


,customer_id,month,revenue,plan
0,C101,2026-01-01,1000,BASIC
1,C101,2026-02-01,1200,BASIC
2,C101,2026-03-01,1500,PRO
3,C101,2026-04-01,1400,PRO
4,C102,2026-01-01,2000,PRO


In [3]:
df.shape


(10, 4)

In [4]:
df.columns

Index(['customer_id', 'month', 'revenue', 'plan'], dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  10 non-null     str  
 1   month        10 non-null     str  
 2   revenue      10 non-null     int64
 3   plan         10 non-null     str  
dtypes: int64(1), str(3)
memory usage: 637.0 bytes


In [7]:
df.dtypes

customer_id      str
month            str
revenue        int64
plan             str
dtype: object

In [9]:
#Converts month to datetime.
df["month"] = pd.to_datetime(df["month"],errors= "coerce")
df.dtypes

customer_id               str
month          datetime64[us]
revenue                 int64
plan                      str
dtype: object

In [10]:
df.head()

,customer_id,month,revenue,plan
0,C101,2026-01-01,1000,BASIC
1,C101,2026-02-01,1200,BASIC
2,C101,2026-03-01,1500,PRO
3,C101,2026-04-01,1400,PRO
4,C102,2026-01-01,2000,PRO


In [37]:
#Ensures records are correctly ordered by customer and month.
df = df.sort_values(["customer_id","month"])

In [38]:
#Creates previous_revenue.
df["previous_revenue"] = df.groupby("customer_id")["revenue"].shift(1)

In [15]:
df.head(3)

,customer_id,month,revenue,plan,previous_revenue
0,C101,2026-01-01,1000,BASIC,NaN
1,C101,2026-02-01,1200,BASIC,1000.0
2,C101,2026-03-01,1500,PRO,1200.0


In [16]:
#Creates revenue_change representing current revenue minus previous revenue.
df["revenue_change"] = df.groupby("customer_id")["revenue"].diff()
df.head(3)


,customer_id,month,revenue,plan,previous_revenue,revenue_change
0,C101,2026-01-01,1000,BASIC,NaN,NaN
1,C101,2026-02-01,1200,BASIC,1000.0,200.0
2,C101,2026-03-01,1500,PRO,1200.0,300.0


In [36]:
#Creates revenue_pct_change as a percentage.

df["revenue_pct_change"] = df.groupby("customer_id")["revenue"].pct_change() * 100
df.head(3)

,customer_id,month,revenue,plan,previous_revenue,revenue_change,revenue_pct_change,cumulative_revenue,subscription_sequence
0,C101,2026-01-01,1000,BASIC,NaN,NaN,NaN,1000,1
1,C101,2026-02-01,1200,BASIC,1000.0,200.0,20.0,2200,2
2,C101,2026-03-01,1500,PRO,1200.0,300.0,25.0,3700,3


In [19]:
#Creates cumulative_revenue separately for each customer.
df["cumulative_revenue"] = df.groupby("customer_id")["revenue"].cumsum()
df.head()

,customer_id,month,revenue,plan,previous_revenue,revenue_change,revenue_pct_change,cumulative_revenue
0,C101,2026-01-01,1000,BASIC,NaN,NaN,NaN,1000
1,C101,2026-02-01,1200,BASIC,1000.0,200.0,0.200000,2200
2,C101,2026-03-01,1500,PRO,1200.0,300.0,0.250000,3700
3,C101,2026-04-01,1400,PRO,1500.0,-100.0,-0.066667,5100
4,C102,2026-01-01,2000,PRO,NaN,NaN,NaN,2000


In [20]:
#Creates subscription_sequence, starting from 1 for every customer.

df["subscription_sequence"] = df.groupby("customer_id").cumcount() + 1
df.head()

,customer_id,month,revenue,plan,previous_revenue,revenue_change,revenue_pct_change,cumulative_revenue,subscription_sequence
0,C101,2026-01-01,1000,BASIC,NaN,NaN,NaN,1000,1
1,C101,2026-02-01,1200,BASIC,1000.0,200.0,0.200000,2200,2
2,C101,2026-03-01,1500,PRO,1200.0,300.0,0.250000,3700,3
3,C101,2026-04-01,1400,PRO,1500.0,-100.0,-0.066667,5100,4
4,C102,2026-01-01,2000,PRO,NaN,NaN,NaN,2000,1


In [21]:
#Find all records where revenue decreased compared with the previous month.
df[df["revenue"] < df["previous_revenue"]]

,customer_id,month,revenue,plan,previous_revenue,revenue_change,revenue_pct_change,cumulative_revenue,subscription_sequence
3,C101,2026-04-01,1400,PRO,1500.0,-100.0,-0.066667,5100,4
5,C102,2026-02-01,1800,PRO,2000.0,-200.0,-0.100000,3800,2


In [35]:
#Find the customer with the highest cumulative revenue at the end of the available period.
df.groupby("customer_id")["revenue"].sum().nlargest(1)

customer_id
C102    6000
Name: revenue, dtype: int64